In [1]:
# A. Feature Engineering
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas11_MLlib").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca dataset
df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

# 1. StringIndexer untuk kolom kategorikal
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Terapkan bertahap
df_t = indexer_menu.fit(df_tugas).transform(df_tugas)
df_t = indexer_cabang.fit(df_t).transform(df_t)
df_t = indexer_waktu.fit(df_t).transform(df_t)

# 2. VectorAssembler menggabungkan seluruh fitur
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_features = assembler.transform(df_t)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 03:48:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
# B. Train/Test Split & Pelatihan Model
from pyspark.ml.regression import LinearRegression

# Split data: 75% training, 25% testing (seed=101)
train_t, test_t = df_features.randomSplit([0.75, 0.25], seed=101)

# Latih Linear Regression
lr_tugas = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_tugas = lr_tugas.fit(train_t)

print("Model Tugas Mandiri berhasil dilatih!")

Model Tugas Mandiri berhasil dilatih!


In [4]:
# C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator

pred_tugas = model_tugas.transform(test_t)

eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_val = eval_rmse.evaluate(pred_tugas)
r2_val = eval_r2.evaluate(pred_tugas)

print(f"RMSE : Rp {rmse_val:,.0f}")
print(f"R²   : {r2_val:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


Model memiliki nilai $R^2$ yang sangat tinggi karena variabel target (total_penjualan) diturunkan langsung dari perkalian qty dan harga_satuan ditambah gangguan Gaussian kecil. Model ini sangat akurat dan layak digunakan untuk membantu perencanaan stok omzet harian.

In [5]:
# D. Eksperimen Perbandingan Fitur
# Model Kedua: Hanya menggunakan fitur 'qty' dan 'harga_satuan'
assembler_2 = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_2")
df_features_2 = assembler_2.transform(df_tugas)

train_2, test_2 = df_features_2.randomSplit([0.75, 0.25], seed=101)

model_2 = LinearRegression(featuresCol="features_2", labelCol="total_penjualan").fit(train_2)
pred_2 = model_2.transform(test_2)

r2_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_2)
print(f"R² Model Kedua (Tanpa fitur kategori): {r2_2:.4f}")

R² Model Kedua (Tanpa fitur kategori): 0.9108


Menambahkan informasi menu, cabang, dan waktu tidak memberikan lonjakan akurasi yang signifikan karena variabel tersebut bersifat acak dan tidak memengaruhi formula dasar pendapatan.

In [6]:
# E. Prediksi Skenario Baru
from pyspark.sql import Row

# 1. Buat DataFrame skenario baru (3 data dummy)
data_skenario = [
    Row(nama_menu="Kopi Susu", cabang="Magelang", waktu_pesan="Pagi", qty=2, harga_satuan=20000),
    Row(nama_menu="Cappuccino", cabang="Yogyakarta", waktu_pesan="Malam", qty=4, harga_satuan=25000),
    Row(nama_menu="Kentang Goreng", cabang="Semarang", waktu_pesan="Sore", qty=3, harga_satuan=18000)
]
df_skenario_raw = spark.createDataFrame(data_skenario)

# 2. Transform menggunakan model indexer yang SUDAH ADA (jangan fit() ulang!)
df_s1 = indexer_menu.fit(df_tugas).transform(df_skenario_raw)
df_s2 = indexer_cabang.fit(df_tugas).transform(df_s1)
df_s3 = indexer_waktu.fit(df_tugas).transform(df_s2)

# 3. Assembler fitur
df_s_final = assembler.transform(df_s3)

# 4. Lakukan Prediksi
hasil_prediksi_skenario = model_tugas.transform(df_s_final)
hasil_prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

[Stage 28:===================>                                      (1 + 2) / 3]

+--------------+----------+-----------+---+------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+--------------+----------+-----------+---+------------+------------------+
|Kopi Susu     |Magelang  |Pagi       |2  |20000       |37525.351984419525|
|Cappuccino    |Yogyakarta|Malam      |4  |25000       |97228.99193122772 |
|Kentang Goreng|Semarang  |Sore       |3  |18000       |54610.801930380665|
+--------------+----------+-----------+---+------------+------------------+



In [8]:
spark.stop()